# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring a FAIR^2 dataset using the `mlcroissant` library. It demonstrates how to load the dataset metadata and records, examine available record sets and fields, extract and process tabular data, and visualize key features.

### Dataset Source
The dataset source is provided via a Croissant schema URL. The schema describes tabulated clinical, hormonal, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Install mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. 

The dataset consists of multiple record sets, each structured as a table:
- Table 1: Clinical Features
- Table 2: Hormone Levels, Imaging Results
- Table 3: Genetic Variant Information

We'll query the record sets and display their identifiers (`@id`) and their available fields.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets()

print('Available record sets and their field IDs:')
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs['name']} | Description: {rs['description']}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']} | Name: {field['name']} | DataType: {field.get('dataType')}")
    print()

# For example, print the first few rows from each record_set
for rs in record_sets:
    print(f"Sample records from {rs['name']} (@id={rs['@id']}):")
    for i, record in enumerate(dataset.records(record_set=rs['@id'])):
        print(record)
        if i >= 1:
            break
    print()


## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. We'll reference record sets and their field `@id`s as queried above.

- Each table is loaded using the `@id` for the record set.
- DataFrames are constructed for easier processing.

In [ ]:
# Collect the record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

# Extract and load each record set to a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

# Choose one record set to continue with: e.g., the first
example_record_set_id = record_set_ids[0]
example_df = dataframes[example_record_set_id]


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, grouping. 
- Choose a numeric field (e.g., age or hormone value) by its `@id`.
- Remove records below a threshold.
- Normalize numeric values.
- Group data by a categorical field, if present.

In [ ]:
# Example EDA on Table 1 (Clinical Features)
# Identify a numeric field and a group/categorical field by @id

numeric_field_id = None
group_field_id = None

# Find a field that is numeric (e.g., 'Age at Diagnosis')
fields = dataset.record_sets()[0]['fields']
for field in fields:
    if field.get('dataType', '').lower() in ['integer', 'float', 'number', 'schema:integer', 'schema:float']:
        numeric_field_id = field['@id']
        break
    # Just for demonstration, use first field

for field in fields:
    # Choose a categorical field (e.g., 'Infertility')
    if field.get('dataType', '').lower() in ['boolean', 'text', 'string', 'schema:boolean', 'schema:text']:
        group_field_id = field['@id']
        break

print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Ensure numeric conversion
if numeric_field_id and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10  # For example, age > 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization

Visualize distributions or relationships in the clinical/hormone/genetic data.

- Distribution of age (or hormone value)
- Relationship between clinical features and numeric values

Here, we demonstrate basic plots for the extracted table.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), bins=5, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If group field exists, show relationship
if group_field_id and numeric_field_id and group_field_id in df.columns:
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()


## 6. Conclusion

This notebook demonstrated how to load, process, and visualize a FAIR^2 dataset using the mlcroissant library. The analysis illustrated structured exploration of clinical features, hormone levels, and genetic variants, referenced strictly by their `@id`s as defined in the Croissant schema.

- Data was loaded according to schema definitions using `mlcroissant`.
- Record set and field `@id`s were used throughout to ensure clear traceability.
- Processing steps included filtering, normalization, grouping, and basic visualizations.

This structured approach supports reproducible science and facilitates downstream applications such as clinical illustration, hypothesis generation, and rare disease cohort exploration.

**Note:** Due to the small sample size (N=3, single center), analyses serve as illustrative examples rather than generalizable statistical conclusions.